# Assignment 5, Question 5: Missing Data Analysis

**Points: 15**

Apply and compare different missing data strategies on the clinical trial dataset.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import utilities from Q3
from q3_data_utils import load_data, detect_missing, fill_missing

# Load the data
df = load_data('data/clinical_trial_raw.csv')
print(f"Loaded {len(df)} patients")

# Prewritten visualization function for missing data
def visualize_missing_data(missing_counts):
    """
    Create a bar chart of missing values by column.
    
    Args:
        missing_counts: pandas Series with missing value counts per column
    """
    plt.figure(figsize=(10, 6))
    missing_counts.plot(kind='bar')
    plt.title('Missing Values by Column')
    plt.xticks(rotation=45)
    plt.ylabel('Number of Missing Values')
    plt.tight_layout()
    plt.show()

## Part 1: Detect Missing Data (3 points)

1. Use the `detect_missing()` utility to find missing values
2. Visualize missing data with a bar plot
3. Calculate the percentage of missing values per column

In [ ]:
# TODO: Detect and analyze missing data
from q3_data_utils import detect_missing

df_missing_values = detect_missing(df)

df_missing_values = df.isnull().sum()
missing_counts = df_missing_values[df_missing_values > 0]
missing_percentages = (missing_counts / len(df)) * 100

print("Missing Value Counts:\n", missing_counts)
print("\nMissing Value Percentages (%):\n", missing_percentages)

Missing_counts_percents = visualize_missing_data(missing_counts)
# Steps:

# 1. Use detect_missing(df) to get missing value counts
# 2. Calculate percentage of missing values per column  
# 3. Print both counts and percentages
# 4. Identify which columns have missing data

# Optional: Use the visualization function above to create a bar chart
# visualize_missing_data(missing_counts)


## Part 2: Compare Imputation Strategies (6 points)

For the 'cholesterol_total' column (which has missing values):

1. Fill with mean using `fill_missing()` utility
2. Fill with median using `fill_missing()` utility  
3. Forward fill using pandas `.fillna(method='ffill')`
4. Compare the three strategies - create a summary table showing:
   - Original mean/median
   - Mean/median after each strategy
   - How many values were filled

In [ ]:
# TODO: Compare imputation strategies
col = "cholesterol_total"

SENTINELS = [-999, "NA", "N/A", ""]
df = df.replace(SENTINELS, np.nan)

# --- build a clean BASE series with real NaNs (in case of -999 / "NA" etc.) ---
base = pd.to_numeric(
    df[col].replace([-999, "NA", "N/A", ""], np.nan), errors="coerce"
)

# originals (before any filling)
orig_mean   = base.mean()
orig_median = base.median()
orig_missing = base.isna().sum()

# copies that start from the same base
df_mean   = df.copy();   df_mean[col]   = base
df_median = df.copy();   df_median[col] = base
df_ffill  = df.copy();   df_ffill[col]  = base

# 1) mean imputation (your utility)
df_mean   = fill_missing(df_mean, column=col, strategy="mean")

# 2) median imputation (your utility)
df_median = fill_missing(df_median, column=col, strategy="median")

# 3) forward fill using pandas
df_ffill[col] = df_ffill[col].fillna(method="ffill")

# missing counts after each strategy
mean_missing   = df_mean[col].isna().sum()
median_missing = df_median[col].isna().sum()
ffill_missing  = df_ffill[col].isna().sum()

# summary table
impute_summary = pd.DataFrame({
    "Original Mean":           [orig_mean],
    "Original Median":         [orig_median],
    "Mean Imputed Mean":       [df_mean[col].mean()],
    "Mean Imputed Median":     [df_mean[col].median()],
    "Median Imputed Mean":     [df_median[col].mean()],
    "Median Imputed Median":   [df_median[col].median()],
    "FFill Imputed Mean":      [df_ffill[col].mean()],
    "FFill Imputed Median":    [df_ffill[col].median()],
    "Original Missing Values": [orig_missing],
    "Filled with Mean":        [orig_missing - mean_missing],
    "Filled with Median":      [orig_missing - median_missing],
    "Filled with FFill":       [orig_missing - ffill_missing],
})

print("\nImputation Comparison for 'cholesterol_total':\n", impute_summary)



## Part 3: Dropping Missing Data (3 points)

1. Drop rows where ANY column has missing data - how many rows remain?
2. Drop rows where specific columns have missing data (e.g., only 'age' or 'bmi')
3. Which approach loses less data?

In [ ]:
# TODO: Drop missing rows with different strategies

# 1) Drop rows with any missing values
df_drop_any = df.dropna(how='any')
print(f"\nAfter dropping rows with any missing values: {len(df_drop_any)} patients")
# 2) Drop rows with specific columns missing data
df_drop_specific = df.dropna(subset=['systolic_bp', 'diastolic_bp'])
print(f"After dropping rows with missing 'systolic_bp' or 'diastolic_bp': {len(df_drop_specific)} patients")

## The second strategy retains more patients while ensuring key blood pressure data is present.

## Part 4: Create Clean Dataset (3 points)

Apply your chosen strategy to create a clean dataset:
1. Choose appropriate imputation for numeric columns
2. Drop rows with missing critical values (e.g., patient_id, age)
3. Save to `output/q5_cleaned_data.csv`
4. Save a missing data report to `output/q5_missing_report.txt`

In [ ]:
# TODO: Create and save clean dataset
df_mean_cleaned = fill_missing(df, column=df.select_dtypes(include=[np.number]).columns.tolist(), strategy='mean')
df_drop_critical = df_mean_cleaned.dropna(subset=['patient_id', 'age'])
df_drop_critical.to_csv('output/q5_cleaned_data.csv', sep="\t", float_format="%.0f", index =False)
df_drop_critical.to_csv('output/q5_missing-report.txt', sep="\t", float_format="%.0f", index =False)


## Reflection

Which imputation strategy would you recommend for this dataset and why?

**Your answer:**

TODO: Explain your strategy choice

I recommend using the mean imputation method because the value will be more representative of the study population's numeric values and replace the missing value with a value that won't drastically impact the dataset or shift the distribution. Because this is a larger dataset with a decent amount of missing values, chossing the incorrect imputation strategy can definitely impact the numeric data distribution.